# RRGライフサイクル別のテーマ残差リターン評価

このノートブックは、**RRG（Relative Rotation Graph）で定義したテーマ・ライフサイクル状態ごとに、将来のBarra残差リターンがどう異なるか**を評価するための実装です。

主な目的は次の通りです。

1. RRG状態（Leading / Weakening / Lagging / Improving）ごとの将来残差リターンを評価する。
2. APWCが高いRRG状態は、単なるRRG状態よりも高品質なテーマ局面を識別できるか検証する。
3. Narrative SignalがRRG状態の質、特にImprovingやLeadingの将来リターンを改善するか確認する。
4. RRG状態遷移、Leading entry、False Leading、状態episode、状態年齢効果を評価する。
5. RRGライフサイクルを用いた簡易テーマ戦略を比較する。

このノートブックは、以下の入力CSVを想定します。

```text
theme_residual_returns.csv  # 必須: date, theme_id, residual_return
theme_apwc.csv              # 任意: date, theme_id, APWC or APWC_z
theme_narrative_signal.csv  # 任意: date, theme_id, narrative_signal
theme_rrg.csv               # 任意: date, theme_id, RSRatio, RSMomentum, RRG_state
```

必要なデータが存在しない場合は、synthetic demo data を自動生成して全セルが動くようにしています。
実データで実行する際は、次の `CONFIG.DATA_DIR` とファイル名を調整してください。

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

import statsmodels.api as sm
import statsmodels.formula.api as smf

try:
    from scipy.special import expit
except Exception:
    expit = None

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

print("Libraries loaded.")

## 1. 設定

ここでは、入力ファイル、分析ホライズン、RRG計算パラメータ、出力保存設定を指定します。

重要なパラメータは以下です。

- `FORWARD_HORIZONS`: RRG状態観測後、何営業日先の残差リターンを見るか。
- `RRG_LOOKBACK`: RS-Ratioを標準化するrolling window。
- `RRG_MOM_LAG`: RS-Momentumを計算する差分ラグ。
- `APWC_HIGH_Q`, `NARRATIVE_HIGH_Q`: APWC / Narrative Signal のHigh/Low判定に使う分位点。
- `SAVE_OUTPUTS`: 結果CSVを保存するか。

In [ ]:
@dataclass
class CONFIG:
    DATA_DIR: str = "/mnt/data/rrg_lifecycle_input"

    RESIDUAL_RETURNS_FILE: str = "theme_residual_returns.csv"
    APWC_FILE: str = "theme_apwc.csv"
    NARRATIVE_SIGNAL_FILE: str = "theme_narrative_signal.csv"
    RRG_FILE: str = "theme_rrg.csv"

    DATE_COL: str = "date"
    THEME_COL: str = "theme_id"

    FORWARD_HORIZONS: Tuple[int, ...] = (5, 20, 60, 120)

    # residual momentum / volatility
    RMOM_WINDOWS: Tuple[int, ...] = (20, 60, 120)
    VOL_WINDOWS: Tuple[int, ...] = (20, 60)

    # RRG construction
    RRG_LOOKBACK: int = 120
    RRG_MIN_PERIODS: int = 60
    RRG_MOM_LAG: int = 20
    RRG_HYSTERESIS_BAND: float = 0.0  # 0.1などにすると境界ノイズを減らせる

    # Conditional group thresholds
    APWC_HIGH_Q: float = 0.60
    APWC_LOW_Q: float = 0.40
    NARRATIVE_HIGH_Q: float = 0.60
    NARRATIVE_LOW_Q: float = 0.40

    # Leading quality
    FALSE_LEAD_HORIZON: int = 20
    FALSE_LEAD_MIN_DAYS: int = 5

    # Strategy
    STRATEGY_REBALANCE: str = "D"  # "D" daily, "W-FRI" weekly, "M" monthly
    MAX_THEMES_PER_STRATEGY: Optional[int] = None

    # Regression controls
    # If True, panel regressions include C(theme_id) and C(month). Slower but closer to a fixed-effect panel.
    INCLUDE_FIXED_EFFECTS: bool = False

    # Output
    SAVE_OUTPUTS: bool = False
    OUTPUT_DIR: str = "/mnt/data/rrg_lifecycle_outputs"

CONFIG.DATA_DIR = str(Path(CONFIG.DATA_DIR))
Path(CONFIG.DATA_DIR).mkdir(parents=True, exist_ok=True)
Path(CONFIG.OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(CONFIG)

## 2. ユーティリティ関数

ここでは、後続分析で使う共通関数を定義します。

含まれる処理は以下です。

- 日付・テーマ列の標準化
- rolling z-score
- Newey-West t-stat
- 将来リターン作成
- RRG状態分類
- 条件付き集計
- 簡易イベントスタディ

In [ ]:
def standardize_date_theme(df: pd.DataFrame, date_col: str = "date", theme_col: str = "theme_id") -> pd.DataFrame:
    out = df.copy()
    if date_col not in out.columns:
        raise ValueError(f"Missing date column: {date_col}")
    if theme_col not in out.columns:
        raise ValueError(f"Missing theme column: {theme_col}")
    out[date_col] = pd.to_datetime(out[date_col])
    out[theme_col] = out[theme_col].astype(str)
    out = out.sort_values([theme_col, date_col]).reset_index(drop=True)
    return out

def rolling_zscore(s: pd.Series, window: int, min_periods: Optional[int] = None) -> pd.Series:
    if min_periods is None:
        min_periods = max(20, window // 2)
    mu = s.rolling(window, min_periods=min_periods).mean()
    sig = s.rolling(window, min_periods=min_periods).std()
    return (s - mu) / sig.replace(0, np.nan)

def winsorize_series(s: pd.Series, p: float = 0.01) -> pd.Series:
    lo, hi = s.quantile(p), s.quantile(1 - p)
    return s.clip(lo, hi)

def newey_west_tstat_mean(x: pd.Series, lags: int = 5) -> Tuple[float, float, int]:
    """Return mean and HAC t-stat for H0: mean = 0."""
    x = pd.Series(x).dropna()
    n = len(x)
    if n < 5:
        return np.nan, np.nan, n
    y = x.values
    X = np.ones((n, 1))
    res = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": int(max(1, lags))})
    return float(res.params[0]), float(res.tvalues[0]), n

def make_forward_returns(df: pd.DataFrame, ret_col: str, horizons: Tuple[int, ...],
                         date_col: str = "date", theme_col: str = "theme_id") -> pd.DataFrame:
    """Add forward cumulative simple returns by theme using sum of daily residual returns."""
    out = df.sort_values([theme_col, date_col]).copy()
    for h in horizons:
        out[f"fwd_ret_{h}D"] = (
            out.groupby(theme_col)[ret_col]
            .transform(lambda x: x.shift(-1).rolling(h, min_periods=h).sum().shift(-(h-1)))
        )
    return out

def classify_rrg_state(rsratio: float, rsmom: float, band: float = 0.0) -> str:
    """Basic RRG state classification."""
    if pd.isna(rsratio) or pd.isna(rsmom):
        return np.nan
    if rsratio >= band and rsmom >= band:
        return "Leading"
    if rsratio >= band and rsmom < band:
        return "Weakening"
    if rsratio < band and rsmom < band:
        return "Lagging"
    if rsratio < band and rsmom >= band:
        return "Improving"
    return np.nan

RRG_ORDER = ["Lagging", "Improving", "Leading", "Weakening"]
STATE_ORDER = ["Leading", "Weakening", "Lagging", "Improving"]

def state_summary_by_date(panel: pd.DataFrame, state_col: str, ret_col: str, horizon: int, date_col: str = "date") -> pd.DataFrame:
    """State-level forward return summary using cross-sectional average per date then Newey-West t-stat."""
    rows = []
    for state, sub in panel.dropna(subset=[state_col, ret_col]).groupby(state_col):
        by_date = sub.groupby(date_col)[ret_col].mean().sort_index()
        mean, tstat, n_dates = newey_west_tstat_mean(by_date, lags=horizon)
        raw = sub[ret_col].dropna()
        rows.append({
            "state": state,
            "horizon": horizon,
            "obs": len(raw),
            "dates": n_dates,
            "mean": mean,
            "median": raw.median(),
            "hit_rate": (raw > 0).mean(),
            "vol": by_date.std(),
            "ir_annualized_like": mean / by_date.std() * np.sqrt(252 / horizon) if by_date.std() and by_date.std() > 0 else np.nan,
            "nw_tstat": tstat,
            "p25": raw.quantile(0.25),
            "p75": raw.quantile(0.75),
        })
    return pd.DataFrame(rows).sort_values("state")

def safe_qcut_labels(s: pd.Series, q_low: float, q_high: float, low_label: str = "Low", high_label: str = "High") -> pd.Series:
    out = pd.Series(index=s.index, dtype="object")
    if s.dropna().nunique() < 3:
        out[:] = "Mid"
        return out
    lo = s.quantile(q_low)
    hi = s.quantile(q_high)
    out[s <= lo] = low_label
    out[s >= hi] = high_label
    out[(s > lo) & (s < hi)] = "Mid"
    return out

def run_ols_formula(df: pd.DataFrame, formula: str, cluster_col: Optional[str] = None):
    d = df.dropna().copy()
    if len(d) < 50:
        print("Not enough observations for regression:", len(d))
        return None
    model = smf.ols(formula, data=d)
    if cluster_col and cluster_col in d.columns:
        res = model.fit(cov_type="cluster", cov_kwds={"groups": d[cluster_col]})
    else:
        res = model.fit(cov_type="HC1")
    return res

def plot_cumulative_return(ret: pd.Series, title: str):
    ret = ret.dropna().sort_index()
    if ret.empty:
        print(f"No data to plot: {title}")
        return
    cum = (1 + ret).cumprod()
    plt.figure(figsize=(10, 4))
    plt.plot(cum.index, cum.values)
    plt.title(title)
    plt.ylabel("Cumulative return")
    plt.grid(True, alpha=0.3)
    plt.show()

print("Utility functions defined.")

## 3. データ読込、または synthetic demo data の作成

実データが `CONFIG.DATA_DIR` に存在する場合はそれを読み込みます。  
存在しない場合は、以下の特徴を持つデモデータを生成します。

- 複数テーマの日次Barra残差リターン
- APWC_z
- Narrative Signal
- RRG座標は残差NAVから計算
- 一部テーマでは Narrative Signal と APWC が残差リターンに先行するような弱い構造を入れています

このsynthetic dataはコード動作確認用です。実証結果として解釈しないでください。

In [ ]:
def file_exists(name: str) -> bool:
    return Path(CONFIG.DATA_DIR, name).exists()

def create_synthetic_data(n_themes: int = 12, n_days: int = 800, seed: int = 42):
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range("2019-01-01", periods=n_days)
    themes = [f"Theme_{i:02d}" for i in range(1, n_themes + 1)]

    records = []
    apwc_records = []
    narr_records = []

    global_noise = rng.normal(0, 0.003, size=n_days)

    for j, theme in enumerate(themes):
        narr = np.zeros(n_days)
        apwc_latent = np.zeros(n_days)
        eps = rng.normal(0, 0.006 + 0.0004*j, size=n_days)
        cyc = np.sin(np.linspace(0, 8*np.pi, n_days) + rng.uniform(0, 2*np.pi))
        shock = rng.normal(0, 0.08, size=n_days)
        for t in range(1, n_days):
            narr[t] = 0.96 * narr[t-1] + 0.07 * shock[t] + 0.01 * cyc[t]
            apwc_latent[t] = 0.93 * apwc_latent[t-1] + 0.08 * narr[t-15 if t>=15 else 0] + rng.normal(0, 0.05)
        narr_s = pd.Series(narr)
        apwc_s = pd.Series(apwc_latent)
        narr_z = (narr_s - narr_s.rolling(252, min_periods=60).mean()) / narr_s.rolling(252, min_periods=60).std()
        apwc_z = (apwc_s - apwc_s.rolling(252, min_periods=60).mean()) / apwc_s.rolling(252, min_periods=60).std()
        narr_z = np.nan_to_num(narr_z)
        apwc_z = np.nan_to_num(apwc_z)

        ret = np.zeros(n_days)
        phi = 0.08 + 0.02 * rng.normal()
        beta_n = 0.0007 if j % 3 != 0 else 0.00025
        beta_a = 0.0005 if j % 4 != 0 else 0.0001
        beta_inter = 0.00035 if j % 5 != 0 else 0.0
        for t in range(1, n_days):
            lag_n = narr_z[t-20] if t >= 20 else 0
            lag_a = apwc_z[t-5] if t >= 5 else 0
            ret[t] = phi * ret[t-1] + beta_n * lag_n + beta_a * lag_a + beta_inter * lag_n * max(lag_a, 0) + eps[t] + 0.2*global_noise[t]

        for t, d in enumerate(dates):
            records.append({"date": d, "theme_id": theme, "residual_return": ret[t]})
            apwc_records.append({"date": d, "theme_id": theme, "APWC_z": apwc_z[t], "APWC": 0.05 + 0.02 * apwc_z[t]})
            narr_records.append({"date": d, "theme_id": theme, "narrative_signal": narr_z[t]})

    residual = pd.DataFrame(records)
    apwc = pd.DataFrame(apwc_records)
    narr = pd.DataFrame(narr_records)
    return residual, apwc, narr

if file_exists(CONFIG.RESIDUAL_RETURNS_FILE):
    residual = pd.read_csv(Path(CONFIG.DATA_DIR, CONFIG.RESIDUAL_RETURNS_FILE))
    print("Loaded residual returns:", residual.shape)
else:
    residual, apwc, narr = create_synthetic_data()
    print("Created synthetic data.")

if file_exists(CONFIG.APWC_FILE):
    apwc = pd.read_csv(Path(CONFIG.DATA_DIR, CONFIG.APWC_FILE))
    print("Loaded APWC:", apwc.shape)
elif "apwc" not in globals():
    apwc = None

if file_exists(CONFIG.NARRATIVE_SIGNAL_FILE):
    narr = pd.read_csv(Path(CONFIG.DATA_DIR, CONFIG.NARRATIVE_SIGNAL_FILE))
    print("Loaded narrative signal:", narr.shape)
elif "narr" not in globals():
    narr = None

if file_exists(CONFIG.RRG_FILE):
    rrg_external = pd.read_csv(Path(CONFIG.DATA_DIR, CONFIG.RRG_FILE))
    print("Loaded external RRG:", rrg_external.shape)
else:
    rrg_external = None

residual = standardize_date_theme(residual, CONFIG.DATE_COL, CONFIG.THEME_COL)
display(residual.head())

## 4. 分析用パネルの構築

ここでは、残差リターンを中核にして、APWC、Narrative Signal、RRGデータを結合します。

- `APWC_z` がない場合は、`APWC` からテーマ別rolling z-scoreを作ります。
- `narrative_signal` がない場合は0として扱います。
- `theme_rrg.csv` がある場合はそれを使います。
- ない場合は、Barra残差リターンからRRG座標を構築します。

In [ ]:
panel = residual.copy()

# Merge APWC
if apwc is not None:
    apwc = standardize_date_theme(apwc, CONFIG.DATE_COL, CONFIG.THEME_COL)
    use_cols = [CONFIG.DATE_COL, CONFIG.THEME_COL] + [c for c in ["APWC", "APWC_z", "apwc", "apwc_z"] if c in apwc.columns]
    apwc_sub = apwc[use_cols].copy()
    if "apwc_z" in apwc_sub.columns and "APWC_z" not in apwc_sub.columns:
        apwc_sub = apwc_sub.rename(columns={"apwc_z": "APWC_z"})
    if "apwc" in apwc_sub.columns and "APWC" not in apwc_sub.columns:
        apwc_sub = apwc_sub.rename(columns={"apwc": "APWC"})
    panel = panel.merge(apwc_sub, on=[CONFIG.DATE_COL, CONFIG.THEME_COL], how="left")
else:
    panel["APWC"] = np.nan
    panel["APWC_z"] = np.nan

if "APWC_z" not in panel.columns:
    panel["APWC_z"] = np.nan

if panel["APWC_z"].isna().all() and "APWC" in panel.columns and not panel["APWC"].isna().all():
    panel["APWC_z"] = (
        panel.sort_values([CONFIG.THEME_COL, CONFIG.DATE_COL])
        .groupby(CONFIG.THEME_COL)["APWC"]
        .transform(lambda x: rolling_zscore(x, 252, 126))
    )

# Merge Narrative Signal
if narr is not None:
    narr = standardize_date_theme(narr, CONFIG.DATE_COL, CONFIG.THEME_COL)
    sig_candidates = [c for c in ["narrative_signal", "NarrativeSignal", "signal", "narr_signal"] if c in narr.columns]
    if not sig_candidates:
        raise ValueError("Narrative signal file must contain narrative_signal or similar column.")
    sig_col = sig_candidates[0]
    narr_sub = narr[[CONFIG.DATE_COL, CONFIG.THEME_COL, sig_col]].rename(columns={sig_col: "narrative_signal"})
    panel = panel.merge(narr_sub, on=[CONFIG.DATE_COL, CONFIG.THEME_COL], how="left")
else:
    panel["narrative_signal"] = 0.0

panel["narrative_signal"] = panel["narrative_signal"].fillna(0.0)

# Clean residual return
if "residual_return" not in panel.columns:
    alt = [c for c in ["resid_return", "bottom_up_residual_return", "ret"] if c in panel.columns]
    if not alt:
        raise ValueError("Residual return column not found.")
    panel = panel.rename(columns={alt[0]: "residual_return"})

panel["residual_return"] = pd.to_numeric(panel["residual_return"], errors="coerce")
panel = panel.dropna(subset=["residual_return"]).sort_values([CONFIG.THEME_COL, CONFIG.DATE_COL]).reset_index(drop=True)

display(panel.head())
print(panel.shape)

## 5. Residual Momentum、残差ボラティリティ、RRG座標の作成

ここでは、テーマ残差リターンから次を作ります。

- Residual NAV
- Residual Momentum
- Residual Volatility
- RS-Ratio
- RS-Momentum
- RRG State

`theme_rrg.csv` がある場合は外部RRGを優先します。ない場合は、残差リターンからRRGを作ります。

In [ ]:
# Residual NAV / log cumulative residual return
panel["resid_log_nav"] = (
    panel.groupby(CONFIG.THEME_COL)["residual_return"]
    .cumsum()
)

# Benchmark residual NAV: cross-sectional average residual return
bench = (
    panel.groupby(CONFIG.DATE_COL)["residual_return"]
    .mean()
    .rename("bench_residual_return")
    .reset_index()
)
bench["bench_log_nav"] = bench["bench_residual_return"].cumsum()
panel = panel.merge(bench[[CONFIG.DATE_COL, "bench_log_nav"]], on=CONFIG.DATE_COL, how="left")

panel["relative_strength_raw"] = panel["resid_log_nav"] - panel["bench_log_nav"]

# Residual momentum and volatility
for w in CONFIG.RMOM_WINDOWS:
    panel[f"RMOM_{w}D"] = (
        panel.groupby(CONFIG.THEME_COL)["residual_return"]
        .transform(lambda x: x.shift(1).rolling(w, min_periods=max(5, w//2)).sum())
    )
    panel[f"RMOM_{w}D_z"] = (
        panel.groupby(CONFIG.THEME_COL)[f"RMOM_{w}D"]
        .transform(lambda x: rolling_zscore(x, 252, 126))
    )

for w in CONFIG.VOL_WINDOWS:
    panel[f"resid_vol_{w}D"] = (
        panel.groupby(CONFIG.THEME_COL)["residual_return"]
        .transform(lambda x: x.shift(1).rolling(w, min_periods=max(5, w//2)).std())
    )

# RRG
if rrg_external is not None:
    rrg_external = standardize_date_theme(rrg_external, CONFIG.DATE_COL, CONFIG.THEME_COL)
    rrg_cols = [CONFIG.DATE_COL, CONFIG.THEME_COL] + [c for c in ["RSRatio", "RSMomentum", "RRG_state", "rrg_state"] if c in rrg_external.columns]
    rrg_sub = rrg_external[rrg_cols].copy()
    if "rrg_state" in rrg_sub.columns and "RRG_state" not in rrg_sub.columns:
        rrg_sub = rrg_sub.rename(columns={"rrg_state": "RRG_state"})
    panel = panel.merge(rrg_sub, on=[CONFIG.DATE_COL, CONFIG.THEME_COL], how="left")
else:
    panel["RSRatio"] = (
        panel.groupby(CONFIG.THEME_COL)["relative_strength_raw"]
        .transform(lambda x: rolling_zscore(x, CONFIG.RRG_LOOKBACK, CONFIG.RRG_MIN_PERIODS))
    )
    panel["RSMomentum_raw"] = panel.groupby(CONFIG.THEME_COL)["RSRatio"].transform(lambda x: x - x.shift(CONFIG.RRG_MOM_LAG))
    panel["RSMomentum"] = (
        panel.groupby(CONFIG.THEME_COL)["RSMomentum_raw"]
        .transform(lambda x: rolling_zscore(x, CONFIG.RRG_LOOKBACK, CONFIG.RRG_MIN_PERIODS))
    )
    panel["RRG_state"] = panel.apply(
        lambda r: classify_rrg_state(r["RSRatio"], r["RSMomentum"], CONFIG.RRG_HYSTERESIS_BAND),
        axis=1
    )

# Additional RRG geometry
panel["RRG_velocity"] = np.sqrt(
    panel.groupby(CONFIG.THEME_COL)["RSRatio"].transform(lambda x: x.diff(CONFIG.RRG_MOM_LAG)).pow(2)
    + panel.groupby(CONFIG.THEME_COL)["RSMomentum"].transform(lambda x: x.diff(CONFIG.RRG_MOM_LAG)).pow(2)
)
panel["RRG_angle"] = np.arctan2(panel["RSMomentum"], panel["RSRatio"])
panel["RRG_angle_change"] = panel.groupby(CONFIG.THEME_COL)["RRG_angle"].transform(lambda x: x.diff(CONFIG.RRG_MOM_LAG))
panel["distance_to_leading"] = np.maximum(0, -panel["RSRatio"]) + np.maximum(0, -panel["RSMomentum"])

# APWC and narrative cross-sectional high/low flags by date
panel["APWC_group"] = panel.groupby(CONFIG.DATE_COL)["APWC_z"].transform(
    lambda x: safe_qcut_labels(x, CONFIG.APWC_LOW_Q, CONFIG.APWC_HIGH_Q)
)
panel["Narrative_group"] = panel.groupby(CONFIG.DATE_COL)["narrative_signal"].transform(
    lambda x: safe_qcut_labels(x, CONFIG.NARRATIVE_LOW_Q, CONFIG.NARRATIVE_HIGH_Q)
)

display(panel[[CONFIG.DATE_COL, CONFIG.THEME_COL, "residual_return", "APWC_z", "narrative_signal", "RSRatio", "RSMomentum", "RRG_state"]].head(10))
print(panel["RRG_state"].value_counts(dropna=False))

## 6. 将来残差リターンの作成

RRG状態別の評価では、**同時点リターン**ではなく、状態を観測した後の **将来残差リターン** を使います。

ここでは、5D / 20D / 60D / 120D の累積残差リターンを作ります。

\[
R^{resid}_{g,t+1:t+h} = \sum_{	au=t+1}^{t+h} \epsilon_{g,	au}
\]

In [ ]:
panel = make_forward_returns(panel, "residual_return", CONFIG.FORWARD_HORIZONS, CONFIG.DATE_COL, CONFIG.THEME_COL)

fwd_cols = [f"fwd_ret_{h}D" for h in CONFIG.FORWARD_HORIZONS]
display(panel[[CONFIG.DATE_COL, CONFIG.THEME_COL, "RRG_state"] + fwd_cols].dropna().head())

## 7. RRG状態別の将来残差リターン評価

最初に、各RRG状態ごとの将来残差リターンを集計します。

評価する統計量は以下です。

- 平均
- 中央値
- Hit rate
- ボラティリティ
- Information ratio
- Newey-West t-stat

ここでは、同一日付内のテーマ横断平均を取ってから、時系列方向にNewey-West t-statを計算します。これにより、テーマ横断の同時相関とforward returnの重複の影響をある程度抑えます。

In [ ]:
state_summaries = []
for h in CONFIG.FORWARD_HORIZONS:
    ret_col = f"fwd_ret_{h}D"
    s = state_summary_by_date(panel, "RRG_state", ret_col, h, CONFIG.DATE_COL)
    state_summaries.append(s)
state_summary = pd.concat(state_summaries, ignore_index=True)

display(state_summary)

# Plot 20D and 60D means
for h in [20, 60]:
    if h in CONFIG.FORWARD_HORIZONS:
        tmp = state_summary[state_summary["horizon"] == h].copy()
        plt.figure(figsize=(8, 4))
        plt.bar(tmp["state"], tmp["mean"])
        plt.axhline(0, color="black", linewidth=1)
        plt.title(f"Forward residual return by RRG state: {h}D")
        plt.ylabel("Mean forward residual return")
        plt.grid(True, axis="y", alpha=0.3)
        plt.show()

## 8. RRG状態ダミーのパネル回帰

状態別平均だけでは、テーマ固有の癖や時点効果を十分にコントロールできません。  
ここではLaggingを基準にして、RRG状態ダミーが将来残差リターンを説明するかをパネル回帰で確認します。

基本モデルは次です。

\[
R^{resid}_{g,t+1:t+h}
=
lpha_g + \lambda_m
+ eta_1 Leading_{g,t}
+ eta_2 Weakening_{g,t}
+ eta_3 Improving_{g,t}
+ Controls_{g,t}
+ u_{g,t+h}
\]

月固定効果とテーマ固定効果を入れています。

In [ ]:
reg_panel = panel.copy()
reg_panel["month"] = reg_panel[CONFIG.DATE_COL].dt.to_period("M").astype(str)

# Make state dummies with Lagging as baseline
reg_panel["state_Leading"] = (reg_panel["RRG_state"] == "Leading").astype(int)
reg_panel["state_Weakening"] = (reg_panel["RRG_state"] == "Weakening").astype(int)
reg_panel["state_Improving"] = (reg_panel["RRG_state"] == "Improving").astype(int)

reg_results_state = {}
for h in [20, 60]:
    if f"fwd_ret_{h}D" not in reg_panel.columns:
        continue
    base_formula = (
        f"fwd_ret_{h}D ~ state_Leading + state_Weakening + state_Improving "
        f"+ APWC_z + narrative_signal + RMOM_60D + resid_vol_60D + RSRatio + RSMomentum"
    )
    if CONFIG.INCLUDE_FIXED_EFFECTS:
        formula = base_formula + f" + C({CONFIG.THEME_COL}) + C(month)"
    else:
        formula = base_formula
    d = reg_panel.dropna(subset=[f"fwd_ret_{h}D", "RRG_state", "APWC_z", "narrative_signal", "RMOM_60D", "resid_vol_60D", "RSRatio", "RSMomentum", "month"]).copy()
    if len(d) > 100:
        res = smf.ols(formula, data=d).fit(cov_type="cluster", cov_kwds={"groups": d[CONFIG.THEME_COL]})
        reg_results_state[h] = res
        print(f"\n===== Panel regression: forward {h}D residual return =====")
        display(res.summary().tables[1])
    else:
        print(f"Not enough observations for {h}D.")

## 9. RRG状態 × APWC High/Low

次に、RRG状態の「質」をAPWCで分けて評価します。

仮説は次です。

\[
E[R^{resid}_{future} \mid RRGState, APWCHigh]
>
E[R^{resid}_{future} \mid RRGState, APWCLow]
\]

特に重要なのは、**Improving × APWC High** と **Leading × APWC High** です。

In [ ]:
def conditional_forward_table(df: pd.DataFrame, group_cols: List[str], ret_col: str, horizon: int) -> pd.DataFrame:
    rows = []
    d = df.dropna(subset=group_cols + [ret_col]).copy()
    for keys, sub in d.groupby(group_cols):
        if not isinstance(keys, tuple):
            keys = (keys,)
        by_date = sub.groupby(CONFIG.DATE_COL)[ret_col].mean().sort_index()
        mean, tstat, n_dates = newey_west_tstat_mean(by_date, lags=horizon)
        raw = sub[ret_col].dropna()
        row = {col: key for col, key in zip(group_cols, keys)}
        row.update({
            "horizon": horizon,
            "obs": len(raw),
            "dates": n_dates,
            "mean": mean,
            "median": raw.median(),
            "hit_rate": (raw > 0).mean(),
            "nw_tstat": tstat,
        })
        rows.append(row)
    return pd.DataFrame(rows)

apwc_cond_tables = []
for h in [20, 60]:
    ret_col = f"fwd_ret_{h}D"
    tbl = conditional_forward_table(panel, ["RRG_state", "APWC_group"], ret_col, h)
    apwc_cond_tables.append(tbl)
    print(f"\n===== RRG state x APWC group, fwd {h}D =====")
    display(tbl.sort_values(["RRG_state", "APWC_group"]))

apwc_cond_table = pd.concat(apwc_cond_tables, ignore_index=True)

## 10. RRG状態 × Narrative Signal High/Low

同様に、RRG状態をNarrative SignalのHigh/Lowで分けます。

仮説は次です。

\[
E[R^{resid}_{future} \mid RRGState, NarrativeHigh]
>
E[R^{resid}_{future} \mid RRGState, NarrativeLow]
\]

特に **Improving × Narrative High** が早期エントリー候補になり得ます。

In [ ]:
narr_cond_tables = []
for h in [20, 60]:
    ret_col = f"fwd_ret_{h}D"
    tbl = conditional_forward_table(panel, ["RRG_state", "Narrative_group"], ret_col, h)
    narr_cond_tables.append(tbl)
    print(f"\n===== RRG state x Narrative group, fwd {h}D =====")
    display(tbl.sort_values(["RRG_state", "Narrative_group"]))

narr_cond_table = pd.concat(narr_cond_tables, ignore_index=True)

## 11. RRG状態 × APWC × Narrative Signal

ここが実務上最も重要な条件付き評価です。

特に見たいセルは次です。

\[
RRGState=Improving,\quad APWC=High,\quad Narrative=High
\]

これは、**RRG上はまだLeading入り前だが、テーマ内残差共振とニュース上の追い風が両方ある状態**です。

In [ ]:
triple_cond_tables = []
for h in [20, 60]:
    ret_col = f"fwd_ret_{h}D"
    tbl = conditional_forward_table(panel, ["RRG_state", "APWC_group", "Narrative_group"], ret_col, h)
    triple_cond_tables.append(tbl)
    print(f"\n===== RRG state x APWC group x Narrative group, fwd {h}D =====")
    display(tbl.sort_values(["RRG_state", "APWC_group", "Narrative_group"]))

triple_cond_table = pd.concat(triple_cond_tables, ignore_index=True)

## 12. RRG状態 × APWC × Narrative の相互作用回帰

条件付き平均の次に、回帰で相互作用を検証します。

中心仮説は以下です。

\[
Improving 	imes APWCZ 	imes NarrativeSignal > 0
\]

つまり、Improving状態でAPWCとNarrative Signalが高いテーマほど、将来残差リターンが高いかを見ます。

In [ ]:
reg_panel["is_Leading"] = (reg_panel["RRG_state"] == "Leading").astype(int)
reg_panel["is_Improving"] = (reg_panel["RRG_state"] == "Improving").astype(int)
reg_panel["is_Weakening"] = (reg_panel["RRG_state"] == "Weakening").astype(int)

# interaction terms
reg_panel["APWC_x_Narr"] = reg_panel["APWC_z"] * reg_panel["narrative_signal"]
reg_panel["Improving_x_APWC"] = reg_panel["is_Improving"] * reg_panel["APWC_z"]
reg_panel["Improving_x_Narr"] = reg_panel["is_Improving"] * reg_panel["narrative_signal"]
reg_panel["Improving_x_APWC_x_Narr"] = reg_panel["is_Improving"] * reg_panel["APWC_z"] * reg_panel["narrative_signal"]
reg_panel["Leading_x_APWC_x_Narr"] = reg_panel["is_Leading"] * reg_panel["APWC_z"] * reg_panel["narrative_signal"]

reg_results_interaction = {}
for h in [20, 60]:
    base_formula = (
        f"fwd_ret_{h}D ~ is_Leading + is_Improving + is_Weakening "
        f"+ APWC_z + narrative_signal + RMOM_60D + APWC_x_Narr "
        f"+ Improving_x_APWC + Improving_x_Narr + Improving_x_APWC_x_Narr "
        f"+ Leading_x_APWC_x_Narr "
        f"+ RSRatio + RSMomentum + resid_vol_60D"
    )
    if CONFIG.INCLUDE_FIXED_EFFECTS:
        formula = base_formula + f" + C({CONFIG.THEME_COL}) + C(month)"
    else:
        formula = base_formula
    needed = [f"fwd_ret_{h}D", "APWC_z", "narrative_signal", "RMOM_60D", "RSRatio", "RSMomentum", "resid_vol_60D", "month"]
    d = reg_panel.dropna(subset=needed).copy()
    if len(d) > 100:
        res = smf.ols(formula, data=d).fit(cov_type="cluster", cov_kwds={"groups": d[CONFIG.THEME_COL]})
        reg_results_interaction[h] = res
        print(f"\n===== Interaction regression: forward {h}D =====")
        display(res.summary().tables[1])
    else:
        print(f"Not enough observations for interaction regression {h}D.")

## 13. RRG状態遷移ごとの将来残差リターン

RRG状態のライフサイクルをより直接見るため、状態遷移ごとの将来残差リターンを評価します。

例：

- Lagging → Improving: 底打ち
- Improving → Leading: 上昇テーマ化
- Leading → Weakening: 減速開始
- Weakening → Lagging: テーマ終了

ここでは \(h\) 営業日後の状態を使って遷移を定義します。

In [ ]:
transition_tables = []

for h in [20, 60]:
    tmp = panel.sort_values([CONFIG.THEME_COL, CONFIG.DATE_COL]).copy()
    tmp[f"state_fwd_{h}D"] = tmp.groupby(CONFIG.THEME_COL)["RRG_state"].shift(-h)
    tmp[f"transition_{h}D"] = tmp["RRG_state"].astype(str) + " -> " + tmp[f"state_fwd_{h}D"].astype(str)
    ret_col = f"fwd_ret_{h}D"
    tbl = conditional_forward_table(tmp, [f"transition_{h}D"], ret_col, h)
    transition_tables.append(tbl)
    print(f"\n===== Transition return table, horizon {h}D =====")
    display(tbl.sort_values("mean", ascending=False).head(20))

transition_table = pd.concat(transition_tables, ignore_index=True)

## 14. RRG状態episode分析

日次データでは同じ状態が連続するため、観測数が長期episodeに偏ります。  
ここでは、連続した同一RRG状態を1つのepisodeとして扱い、状態ごとのepisode期間とepisode残差リターンを評価します。

これにより、

- Leadingはどれくらい続くのか
- Improvingは短命なのか
- WeakeningからLaggingへ移りやすいのか
- 状態別のepisode returnはどう違うのか

を確認できます。

In [ ]:
def make_state_episodes(df: pd.DataFrame, state_col: str = "RRG_state") -> pd.DataFrame:
    rows = []
    for theme, sub in df.sort_values([CONFIG.THEME_COL, CONFIG.DATE_COL]).groupby(CONFIG.THEME_COL):
        sub = sub.dropna(subset=[state_col]).copy()
        if sub.empty:
            continue
        change = sub[state_col].ne(sub[state_col].shift()).cumsum()
        sub["episode_id"] = change
        for eid, e in sub.groupby("episode_id"):
            state = e[state_col].iloc[0]
            start = e[CONFIG.DATE_COL].iloc[0]
            end = e[CONFIG.DATE_COL].iloc[-1]
            duration = len(e)
            ep_ret = (1 + e["residual_return"]).prod() - 1
            ep_sum = e["residual_return"].sum()
            rows.append({
                "theme_id": theme,
                "state": state,
                "episode_id": f"{theme}_{eid}",
                "start_date": start,
                "end_date": end,
                "duration_days": duration,
                "episode_return": ep_ret,
                "episode_sum_return": ep_sum,
                "mean_daily_return": e["residual_return"].mean(),
                "apwc_start": e["APWC_z"].iloc[0] if "APWC_z" in e else np.nan,
                "narr_start": e["narrative_signal"].iloc[0] if "narrative_signal" in e else np.nan,
            })
    return pd.DataFrame(rows)

episodes = make_state_episodes(panel)

episode_summary = (
    episodes.groupby("state")
    .agg(
        episodes=("episode_id", "count"),
        mean_duration=("duration_days", "mean"),
        median_duration=("duration_days", "median"),
        mean_episode_return=("episode_return", "mean"),
        median_episode_return=("episode_return", "median"),
        hit_rate=("episode_return", lambda x: (x > 0).mean()),
        mean_daily_return=("mean_daily_return", "mean"),
    )
    .reset_index()
)

display(episode_summary)

plt.figure(figsize=(8, 4))
plt.bar(episode_summary["state"], episode_summary["mean_episode_return"])
plt.axhline(0, color="black", linewidth=1)
plt.title("Mean episode return by RRG state")
plt.grid(True, axis="y", alpha=0.3)
plt.show()

## 15. 状態年齢効果：Early / Mature / Late Leading

Leadingの初期と末期では、将来リターンの意味が異なる可能性があります。

ここでは各状態に入ってからの経過日数 `state_age` を計算し、Leadingを以下に分けます。

- Early Leading: 20営業日以内
- Mature Leading: 21〜60営業日
- Late Leading: 61営業日以上

Late Leading で将来残差リターンが低下するなら、利食い・縮小シグナルとして使えます。

In [ ]:
def add_state_age(df: pd.DataFrame, state_col: str = "RRG_state") -> pd.DataFrame:
    out = df.sort_values([CONFIG.THEME_COL, CONFIG.DATE_COL]).copy()
    age_series = []
    for theme, sub in out.groupby(CONFIG.THEME_COL):
        prev = None
        cur_age = 0
        for s in sub[state_col]:
            if pd.isna(s):
                cur_age = np.nan
                prev = None
                age_series.append(np.nan)
            else:
                if s == prev:
                    cur_age += 1
                else:
                    cur_age = 1
                    prev = s
                age_series.append(cur_age)
    out["state_age"] = age_series
    return out

panel = add_state_age(panel)
panel["Leading_age_group"] = pd.Series(index=panel.index, dtype="object")
mask_lead = panel["RRG_state"] == "Leading"
panel.loc[mask_lead & (panel["state_age"] <= 20), "Leading_age_group"] = "Early Leading"
panel.loc[mask_lead & (panel["state_age"] > 20) & (panel["state_age"] <= 60), "Leading_age_group"] = "Mature Leading"
panel.loc[mask_lead & (panel["state_age"] > 60), "Leading_age_group"] = "Late Leading"

leading_age_tables = []
for h in [20, 60]:
    ret_col = f"fwd_ret_{h}D"
    tbl = conditional_forward_table(panel.dropna(subset=["Leading_age_group"]), ["Leading_age_group"], ret_col, h)
    leading_age_tables.append(tbl)
    print(f"\n===== Leading age group forward returns, {h}D =====")
    display(tbl.sort_values("Leading_age_group"))

leading_age_table = pd.concat(leading_age_tables, ignore_index=True)

## 16. Leading Entry イベントスタディ

Leading入り日をイベントとして、イベント後の累積Barra残差リターンを評価します。

特に以下の条件別に比較します。

- APWC High / Low
- Narrative High / Low
- APWC High かつ Narrative High

これにより、**APWCとNarrativeの裏付けを伴うLeading入りは、その後の残差リターンが高いか**を確認します。

In [ ]:
event_panel = panel.sort_values([CONFIG.THEME_COL, CONFIG.DATE_COL]).copy()
event_panel["prev_state"] = event_panel.groupby(CONFIG.THEME_COL)["RRG_state"].shift(1)
event_panel["LeadingEntry"] = ((event_panel["RRG_state"] == "Leading") & (event_panel["prev_state"] != "Leading")).astype(int)

event_panel["APWC_high_flag"] = (event_panel["APWC_group"] == "High").astype(int)
event_panel["Narr_high_flag"] = (event_panel["Narrative_group"] == "High").astype(int)
event_panel["ConfirmedLeadEntry"] = ((event_panel["LeadingEntry"] == 1) & (event_panel["APWC_high_flag"] == 1) & (event_panel["Narr_high_flag"] == 1)).astype(int)

def leading_entry_event_study(df: pd.DataFrame, event_col: str, window_pre: int = 20, window_post: int = 60) -> pd.DataFrame:
    events = df[df[event_col] == 1][[CONFIG.DATE_COL, CONFIG.THEME_COL]].copy()
    rows = []
    for _, ev in events.iterrows():
        theme = ev[CONFIG.THEME_COL]
        dt = ev[CONFIG.DATE_COL]
        sub = df[df[CONFIG.THEME_COL] == theme].sort_values(CONFIG.DATE_COL).reset_index(drop=True)
        idx_arr = sub.index[sub[CONFIG.DATE_COL] == dt].tolist()
        if not idx_arr:
            continue
        idx = idx_arr[0]
        lo, hi = idx - window_pre, idx + window_post
        if lo < 0 or hi >= len(sub):
            continue
        path = sub.iloc[lo:hi+1].copy()
        path["event_time"] = np.arange(-window_pre, window_post + 1)
        path["event_date"] = dt
        path["event_theme"] = theme
        path["cum_resid_return"] = path["residual_return"].where(path["event_time"] > 0, 0).cumsum()
        rows.append(path[[CONFIG.DATE_COL, CONFIG.THEME_COL, "event_time", "cum_resid_return", "residual_return", "APWC_z", "narrative_signal"]])
    if not rows:
        return pd.DataFrame()
    return pd.concat(rows, ignore_index=True)

lead_event_path = leading_entry_event_study(event_panel, "LeadingEntry")
confirmed_event_path = leading_entry_event_study(event_panel, "ConfirmedLeadEntry")

for name, path in [("All Leading Entry", lead_event_path), ("Confirmed Leading Entry", confirmed_event_path)]:
    if path.empty:
        print(f"No events for {name}")
        continue
    avg = path.groupby("event_time")["cum_resid_return"].mean()
    plt.figure(figsize=(9, 4))
    plt.plot(avg.index, avg.values)
    plt.axvline(0, color="black", linewidth=1)
    plt.axhline(0, color="black", linewidth=1)
    plt.title(f"Event study: {name}")
    plt.xlabel("Event time")
    plt.ylabel("Average cumulative residual return")
    plt.grid(True, alpha=0.3)
    plt.show()

print("Leading entries:", int(event_panel["LeadingEntry"].sum()))
print("Confirmed leading entries:", int(event_panel["ConfirmedLeadEntry"].sum()))

## 17. False Leading 分析

Leading入りしたものの、すぐにLeading状態から外れるケースを **False Leading** として定義します。

ここでは、Leading入り後 `FALSE_LEAD_HORIZON` 営業日のうち、Leading滞在日数が `FALSE_LEAD_MIN_DAYS` 未満ならFalse Leadingとします。

仮説は以下です。

\[
APWCZ高,\ NarrativeSignal高 \Rightarrow FalseLeading確率低下
\]

In [ ]:
lead_entries = event_panel[event_panel["LeadingEntry"] == 1].copy()
false_rows = []

for _, row in lead_entries.iterrows():
    theme = row[CONFIG.THEME_COL]
    dt = row[CONFIG.DATE_COL]
    sub = panel[panel[CONFIG.THEME_COL] == theme].sort_values(CONFIG.DATE_COL).reset_index(drop=True)
    idxs = sub.index[sub[CONFIG.DATE_COL] == dt].tolist()
    if not idxs:
        continue
    idx = idxs[0]
    horizon = CONFIG.FALSE_LEAD_HORIZON
    if idx + horizon >= len(sub):
        continue
    future = sub.iloc[idx:idx+horizon+1]
    leading_days = (future["RRG_state"] == "Leading").sum()
    false = int(leading_days < CONFIG.FALSE_LEAD_MIN_DAYS)
    false_rows.append({
        "date": dt,
        "theme_id": theme,
        "false_leading": false,
        "leading_days": leading_days,
        "APWC_z": row["APWC_z"],
        "narrative_signal": row["narrative_signal"],
        "RMOM_60D": row.get("RMOM_60D", np.nan),
        "RSRatio": row.get("RSRatio", np.nan),
        "RSMomentum": row.get("RSMomentum", np.nan),
    })

false_leading_df = pd.DataFrame(false_rows)
display(false_leading_df.head())
print("False leading rate:", false_leading_df["false_leading"].mean() if not false_leading_df.empty else np.nan)

if len(false_leading_df) > 50 and false_leading_df["false_leading"].nunique() > 1:
    formula = "false_leading ~ APWC_z + narrative_signal + RMOM_60D + RSRatio + RSMomentum"
    try:
        res_false = smf.logit(formula, data=false_leading_df.dropna()).fit(disp=False)
        display(res_false.summary().tables[1])
    except Exception as e:
        print("False leading logit failed:", e)
else:
    print("Not enough false leading events for logit.")

## 18. RRGライフサイクル戦略の比較

RRG状態とAPWC、Narrative Signalを使った簡易戦略を比較します。

各日 \(t\) のシグナルに基づき、翌営業日のテーマ残差リターンを保有します。

比較する戦略は以下です。

1. Leading Strategy
2. Improving Strategy
3. Improving + APWC High + Narrative High
4. Confirmed Leading: Leading + APWC High + Narrative High
5. Exit-aware Strategy: Improving/LeadingかつAPWC/Narrative/RMOMがプラス

これは売買コストや実装制約を入れないシンプルな残差リターン戦略です。

In [ ]:
strat = panel.sort_values([CONFIG.DATE_COL, CONFIG.THEME_COL]).copy()

# next-day residual return
strat["next_resid_return"] = strat.groupby(CONFIG.THEME_COL)["residual_return"].shift(-1)

# Selection rules
strat["sel_leading"] = (strat["RRG_state"] == "Leading").astype(int)
strat["sel_improving"] = (strat["RRG_state"] == "Improving").astype(int)
strat["sel_improving_confirmed"] = (
    (strat["RRG_state"] == "Improving")
    & (strat["APWC_group"] == "High")
    & (strat["Narrative_group"] == "High")
).astype(int)
strat["sel_confirmed_leading"] = (
    (strat["RRG_state"] == "Leading")
    & (strat["APWC_group"] == "High")
    & (strat["Narrative_group"] == "High")
).astype(int)
strat["sel_exit_aware"] = (
    (strat["RRG_state"].isin(["Improving", "Leading"]))
    & (strat["APWC_z"] > 0)
    & (strat["narrative_signal"] > 0)
    & (strat["RMOM_60D"] > 0)
).astype(int)

strategy_cols = {
    "Leading": "sel_leading",
    "Improving": "sel_improving",
    "Improving_APWC_Narr_High": "sel_improving_confirmed",
    "Confirmed_Leading": "sel_confirmed_leading",
    "Exit_Aware": "sel_exit_aware",
}

strategy_returns = []
for name, sel_col in strategy_cols.items():
    d = strat.dropna(subset=["next_resid_return"]).copy()
    if CONFIG.MAX_THEMES_PER_STRATEGY is not None:
        d["combined_score"] = d["APWC_z"].fillna(0) + d["narrative_signal"].fillna(0) + d["RMOM_60D_z"].fillna(0)
        selected = []
        for dt, sub in d.groupby(CONFIG.DATE_COL):
            ss = sub[sub[sel_col] == 1].sort_values("combined_score", ascending=False).head(CONFIG.MAX_THEMES_PER_STRATEGY)
            selected.append(ss)
        d = pd.concat(selected, ignore_index=True) if selected else pd.DataFrame(columns=d.columns)
        ret = d.groupby(CONFIG.DATE_COL)["next_resid_return"].mean()
    else:
        d_sel = d[d[sel_col] == 1]
        ret = d_sel.groupby(CONFIG.DATE_COL)["next_resid_return"].mean()

    all_dates = pd.Index(sorted(strat[CONFIG.DATE_COL].unique()), name=CONFIG.DATE_COL)
    ret = ret.reindex(all_dates).fillna(0.0)
    strategy_returns.append(ret.rename(name))

strategy_returns = pd.concat(strategy_returns, axis=1)
display(strategy_returns.describe().T)

for col in strategy_returns.columns:
    plot_cumulative_return(strategy_returns[col], f"Strategy cumulative residual return: {col}")

strategy_summary = []
for col in strategy_returns.columns:
    r = strategy_returns[col].dropna()
    mean, tstat, n = newey_west_tstat_mean(r, lags=20)
    strategy_summary.append({
        "strategy": col,
        "daily_mean": mean,
        "annualized_mean_like": mean * 252,
        "daily_vol": r.std(),
        "annualized_vol_like": r.std() * np.sqrt(252),
        "sharpe_like": mean / r.std() * np.sqrt(252) if r.std() > 0 else np.nan,
        "hit_rate": (r > 0).mean(),
        "nw_tstat_daily_mean": tstat,
        "n_days": n,
        "max_drawdown": ((1+r).cumprod() / (1+r).cumprod().cummax() - 1).min()
    })
strategy_summary = pd.DataFrame(strategy_summary)
display(strategy_summary)

## 19. RRG座標ヒートマップ：将来残差リターン

RRG状態の4象限だけでは粗いので、RS-Ratio と RS-Momentum をビン分けし、各領域の将来残差リターンを可視化します。

これにより、LeadingやImprovingの中でも、どの座標領域が将来リターンに結びつきやすいかを確認できます。

In [ ]:
def rrg_heatmap(df: pd.DataFrame, horizon: int = 20, bins: int = 7):
    ret_col = f"fwd_ret_{horizon}D"
    d = df.dropna(subset=["RSRatio", "RSMomentum", ret_col]).copy()
    if len(d) < 100:
        print("Not enough data for heatmap.")
        return None
    d["RS_bin"] = pd.qcut(d["RSRatio"], bins, duplicates="drop")
    d["MOM_bin"] = pd.qcut(d["RSMomentum"], bins, duplicates="drop")
    table = d.pivot_table(index="MOM_bin", columns="RS_bin", values=ret_col, aggfunc="mean")
    plt.figure(figsize=(10, 7))
    plt.imshow(table.values, aspect="auto", origin="lower")
    plt.colorbar(label=f"Mean fwd {horizon}D residual return")
    plt.title(f"RRG coordinate heatmap: fwd {horizon}D residual return")
    plt.xlabel("RS-Ratio bin")
    plt.ylabel("RS-Momentum bin")
    plt.xticks(range(len(table.columns)), [str(c) for c in table.columns], rotation=45, ha="right")
    plt.yticks(range(len(table.index)), [str(i) for i in table.index])
    plt.tight_layout()
    plt.show()
    return table

heatmap_20 = rrg_heatmap(panel, 20, bins=7)
if heatmap_20 is not None:
    display(heatmap_20)

## 20. 主要アウトプットの保存

`CONFIG.SAVE_OUTPUTS = True` にすると、主要な分析結果をCSVとして保存します。

保存先は `CONFIG.OUTPUT_DIR` です。

In [ ]:
if CONFIG.SAVE_OUTPUTS:
    outdir = Path(CONFIG.OUTPUT_DIR)
    outdir.mkdir(parents=True, exist_ok=True)

    panel.to_csv(outdir / "rrg_lifecycle_panel.csv", index=False)
    state_summary.to_csv(outdir / "rrg_state_forward_return_summary.csv", index=False)
    apwc_cond_table.to_csv(outdir / "rrg_state_x_apwc_forward_returns.csv", index=False)
    narr_cond_table.to_csv(outdir / "rrg_state_x_narrative_forward_returns.csv", index=False)
    triple_cond_table.to_csv(outdir / "rrg_state_x_apwc_x_narrative_forward_returns.csv", index=False)
    transition_table.to_csv(outdir / "rrg_transition_forward_returns.csv", index=False)
    episodes.to_csv(outdir / "rrg_state_episodes.csv", index=False)
    episode_summary.to_csv(outdir / "rrg_state_episode_summary.csv", index=False)
    leading_age_table.to_csv(outdir / "leading_age_forward_returns.csv", index=False)
    false_leading_df.to_csv(outdir / "false_leading_events.csv", index=False)
    strategy_returns.to_csv(outdir / "rrg_lifecycle_strategy_returns.csv")
    strategy_summary.to_csv(outdir / "rrg_lifecycle_strategy_summary.csv", index=False)
    if heatmap_20 is not None:
        heatmap_20.to_csv(outdir / "rrg_coordinate_heatmap_20D.csv")

    print(f"Saved outputs to {outdir}")
else:
    print("CONFIG.SAVE_OUTPUTS is False. No CSV files saved.")

# まとめ

このノートブックでは、RRGによるテーマ・ライフサイクル状態ごとのBarra残差リターン評価を実装しました。

特に重要な分析は以下です。

1. **RRG状態別の将来残差リターン**
   - Leading / Improving / Weakening / Lagging のどの状態が将来リターンを持つか。

2. **RRG状態 × APWC**
   - APWCが高いLeadingやImprovingは、単なるRRG状態よりも将来リターンが高いか。

3. **RRG状態 × Narrative Signal**
   - Narrative上の追い風が、RRG状態の将来リターンを改善するか。

4. **RRG状態 × APWC × Narrative Signal**
   - 特に `Improving × APWC High × Narrative High` が早期エントリー候補になるか。

5. **状態遷移とepisode分析**
   - Improving → Leading、Leading → Weakening などのライフサイクル遷移ごとの残差リターンを評価。

6. **Leading entry と False Leading**
   - APWCとNarrativeの裏付けを伴うLeading入りは、その後の残差リターンが高く、ダマシが少ないか。

7. **RRGライフサイクル戦略**
   - Leading単独、Improving単独、Confirmed Leading、Improving confirmed などを比較。

最も注目すべき実証仮説は次です。

\[
E[R^{resid}_{g,t+1:t+h} \mid Improving,\ APWC High,\ Narrative High]
>
E[R^{resid}_{g,t+1:t+h} \mid Improving,\ APWC Low,\ Narrative Low]
\]

これが確認できれば、RRGを単なる可視化ではなく、**APWCとNarrative Signalを用いたテーマ・ライフサイクル別残差リターン予測モデル**として使える可能性が高まります。